# 03 - Validate PC Builder Component Candidates

Run this after `02_clean_pc_components.ipynb`.

It checks whether the candidate component catalog is ready to review for runtime promotion. The notebook does not modify `data/components.json`.

## What this validates

- Required PC Builder categories exist.
- Required compatibility specs are present by category.
- SKUs are unique.
- Prices look plausible for each category.
- Marketplace links are present.
- Suspicious monitor, HDD, PSU, and cooler accessory rows are flagged.
- Reviewable JSON and CSV reports are exported.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import display

DATA_DIR = Path('data')
REPORT_DIR = DATA_DIR / 'cleaning_reports'
CANDIDATE_JSON = DATA_DIR / 'component_candidates.json'
CANDIDATE_CSV = DATA_DIR / 'component_candidates.csv'
VALIDATION_REPORT = REPORT_DIR / 'component_validation_report.json'
VALIDATION_ISSUES = REPORT_DIR / 'component_validation_issues.csv'

REPORT_DIR.mkdir(parents=True, exist_ok=True)

if CANDIDATE_JSON.exists():
    records = json.loads(CANDIDATE_JSON.read_text(encoding='utf-8'))
elif CANDIDATE_CSV.exists():
    frame = pd.read_csv(CANDIDATE_CSV).fillna('')
    records = frame.to_dict(orient='records')
    for item in records:
        for key in ['specs', 'marketplace_links']:
            if isinstance(item.get(key), str):
                try:
                    item[key] = json.loads(item[key]) if item[key] else ([] if key == 'marketplace_links' else {})
                except json.JSONDecodeError:
                    item[key] = [] if key == 'marketplace_links' else {}
else:
    raise FileNotFoundError('Run 02_clean_pc_components.ipynb first so data/component_candidates.json exists.')

components = pd.DataFrame(records)
print('Candidate rows:', len(components))
display(components.head(10))

In [ ]:
REQUIRED_CORE_CATEGORIES = ['cpu', 'motherboard', 'ram', 'gpu', 'ssd', 'hdd', 'psu', 'cooler', 'case']
OPTIONAL_CATEGORIES = ['monitor', 'ups']
ALL_PC_BUILDER_CATEGORIES = REQUIRED_CORE_CATEGORIES + OPTIONAL_CATEGORIES

REQUIRED_SPEC_BY_CATEGORY = {
    'cpu': ['socket'],
    'motherboard': ['socket', 'ram_type'],
    'ram': ['type', 'capacity_gb'],
    'gpu': ['vendor'],
    'ssd': ['capacity_gb'],
    'hdd': ['capacity_gb'],
    'psu': ['wattage_w'],
    'cooler': ['type'],
    'case': ['max_form_factor'],
    'monitor': [],
    'ups': [],
}

PRICE_BOUNDS_IDR = {
    'cpu': (300_000, 20_000_000),
    'motherboard': (500_000, 20_000_000),
    'ram': (150_000, 8_000_000),
    'gpu': (700_000, 60_000_000),
    'ssd': (100_000, 10_000_000),
    'hdd': (200_000, 10_000_000),
    'psu': (150_000, 8_000_000),
    'cooler': (50_000, 8_000_000),
    'case': (150_000, 12_000_000),
    'monitor': (500_000, 30_000_000),
    'ups': (300_000, 20_000_000),
}

SUSPICIOUS_PATTERNS = {
    'monitor': re.compile(r'adapter|adaptor|bracket|mount|wall mount|arm|stand|remote|cable|converter|\btv\b|smart tv|android tv', re.I),
    'hdd': re.compile(r'\bnas\b|qnap|synology|asustor|terra ?master|rackmount|rail|bay|enclosure|docking|external|bracket|nvr', re.I),
    'psu': re.compile(r'cable|adapter|connector|extension|sleeved', re.I),
    'cooler': re.compile(r'mounting|accessor|bracket|thermal paste|vga cooler|ssd cooler|memory cooler', re.I),
}


def as_dict(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str) and value.strip():
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, dict) else {}
        except json.JSONDecodeError:
            return {}
    return {}


def as_links(value, item):
    if isinstance(value, list):
        return value
    if isinstance(value, str) and value.strip():
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass
    links = []
    for marketplace, column in [('enterkomputer', 'product_url'), ('tokopedia', 'tokopedia_url'), ('shopee', 'shopee_url')]:
        url = item.get(column)
        if isinstance(url, str) and url.strip():
            links.append({'marketplace': marketplace, 'url': url.strip()})
    return links

In [ ]:
issues = []
category_counts = components.groupby('category').size().sort_values(ascending=False).to_dict() if len(components) else {}

for category in REQUIRED_CORE_CATEGORIES:
    if category_counts.get(category, 0) == 0:
        issues.append({'sku': '', 'name': '', 'category': category, 'issue': 'required_category_missing', 'detail': 'No candidate rows for required category.'})

if len(components) and 'sku' in components:
    duplicates = components[components.duplicated('sku', keep=False)].sort_values('sku')
    for _, row in duplicates.iterrows():
        issues.append({'sku': row.get('sku', ''), 'name': row.get('name', ''), 'category': row.get('category', ''), 'issue': 'duplicate_sku', 'detail': 'SKU appears more than once.'})

for item in records:
    sku = item.get('sku', '')
    name = item.get('name', '')
    category = item.get('category', '')
    specs = as_dict(item.get('specs'))
    links = as_links(item.get('marketplace_links'), item)
    price = pd.to_numeric(item.get('price_idr'), errors='coerce')

    if category not in ALL_PC_BUILDER_CATEGORIES:
        issues.append({'sku': sku, 'name': name, 'category': category, 'issue': 'category_out_of_scope', 'detail': 'Candidate category is not in PC Builder scope.'})

    if not links:
        issues.append({'sku': sku, 'name': name, 'category': category, 'issue': 'marketplace_link_missing', 'detail': 'No EnterKomputer, Tokopedia, or Shopee URL found.'})

    for spec_key in REQUIRED_SPEC_BY_CATEGORY.get(category, []):
        if spec_key not in specs or specs.get(spec_key) in [None, '', 0]:
            issues.append({'sku': sku, 'name': name, 'category': category, 'issue': 'required_spec_missing', 'detail': spec_key})

    bounds = PRICE_BOUNDS_IDR.get(category)
    if bounds and pd.notna(price):
        low, high = bounds
        if int(price) < low or int(price) > high:
            issues.append({'sku': sku, 'name': name, 'category': category, 'issue': 'price_outlier', 'detail': f'{int(price)} outside {low}-{high}'})

    pattern = SUSPICIOUS_PATTERNS.get(category)
    if pattern and pattern.search(f'{name} {item.get("subcategory", "")}'):
        issues.append({'sku': sku, 'name': name, 'category': category, 'issue': 'suspicious_product_row', 'detail': 'Name/subcategory matches an accessory or out-of-scope pattern.'})

issues_df = pd.DataFrame(issues, columns=['sku', 'name', 'category', 'issue', 'detail'])
print('Category counts:', category_counts)
print('Validation issues:', len(issues_df))
display(issues_df.head(100))

In [ ]:
issue_counts = issues_df.groupby('issue').size().sort_values(ascending=False).to_dict() if len(issues_df) else {}
category_issue_counts = issues_df.groupby(['category', 'issue']).size().reset_index(name='rows').sort_values(['category', 'rows'], ascending=[True, False]) if len(issues_df) else pd.DataFrame(columns=['category', 'issue', 'rows'])

report = {
    'candidate_rows': int(len(records)),
    'category_counts': {str(k): int(v) for k, v in category_counts.items()},
    'required_core_categories': REQUIRED_CORE_CATEGORIES,
    'optional_categories': OPTIONAL_CATEGORIES,
    'issue_counts': {str(k): int(v) for k, v in issue_counts.items()},
    'validation_issue_rows': int(len(issues_df)),
    'ready_for_runtime_promotion': int(len(issues_df)) == 0,
    'outputs': {
        'validation_report': str(VALIDATION_REPORT),
        'validation_issues_csv': str(VALIDATION_ISSUES),
    },
}

VALIDATION_REPORT.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
issues_df.to_csv(VALIDATION_ISSUES, index=False)

print('Wrote:', VALIDATION_REPORT)
print('Wrote:', VALIDATION_ISSUES)
display(category_issue_counts.head(100))

## Promotion rule

Only promote candidates into `data/components.json` after validation has been reviewed. A non-zero issue count is not automatically fatal during early scraping, but it should be understood before runtime replacement.